# Stage 1 — Study Area Setup (Mole Creek)

Prototype-first workflow: build the Stage 1 framework for the Mole Creek
case study before scaling statewide (see README "Workflow / Methodology
Reference"). Loads the Karst Atlas, 1:25,000 geology, and DEM, reprojects
everything to a common CRS, and clips to the Mole Creek boundary produced
by `Bounding_box.ipynb`.

**Verified against the actual data** (2026-09-28): Karst Atlas, geology,
and DEM are all natively **EPSG:28355 (GDA94 / MGA Zone 55)** — not
GDA2020/MGA55 (EPSG:7855) as assumed earlier in planning. Using EPSG:28355
as the common analysis CRS avoids an unnecessary datum transformation
across all three core layers. If DEA land use or another GDA2020 dataset
is added later, reproject *it* to EPSG:28355 to match, rather than
reprojecting everything else.

Vector layers are kept as vectors — only the DEM is rasterised, since it
already is one.

In [ ]:
from pathlib import Path
import os

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.mask import mask
from rasterio.merge import merge
import matplotlib.pyplot as plt

TARGET_CRS = "EPSG:28355"  # GDA94 / MGA Zone 55 -- native CRS of Karst Atlas, geology, and DEM

inpath = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = inpath / "Output_data"
outpath.mkdir(exist_ok=True)

## 1. Mole Creek boundary

Load the boundary produced by `Bounding_box.ipynb` and confirm its CRS.

In [ ]:
mole_creek = gpd.read_file(str(outpath / "Mole_Creek_bounding_box_GDA94.shp"))
print("Mole Creek boundary CRS:", mole_creek.crs)
mole_creek = mole_creek.to_crs(TARGET_CRS)

mole_creek.plot(figsize=(6, 6), edgecolor="black", facecolor="none")
plt.title("Mole Creek study area boundary")
plt.show()

## 2. Karst Atlas — clipped to Mole Creek

Confirmed fields (from the real shapefile, not the workflow doc's
shorthand names): `KCATEGORY`, `KEXPOSED`, `KCOVERED`, `KINTERSTR`,
`KPROXCATCH`, `KDISTCATCH`, `KAVRAIN`, `KLITH`, `KNAME`. `KCATEGORY` is
the field to use for karst intensity reclassification (Stage 2).

In [ ]:
karst = gpd.read_file(str(inpath / "KarstAtlas/Karst_Ver_3_1/Karst_Atlas_v3_1.shp")).to_crs(TARGET_CRS)
karst_mc = gpd.clip(karst, mole_creek)

print(f"Karst polygons in Mole Creek: {len(karst_mc)}")
print(karst_mc["KCATEGORY"].value_counts(dropna=False))

karst_mc.to_file(outpath / "karst_mc_GDA94.gpkg", driver="GPKG")

fig, ax = plt.subplots(figsize=(6, 6))
mole_creek.boundary.plot(ax=ax, color="black")
karst_mc.plot(ax=ax, column="KCATEGORY", legend=True, cmap="viridis")
plt.title("Karst Atlas polygons within Mole Creek, by KCATEGORY")
plt.show()

## 3. Geology — clipped to Mole Creek

The 1:25,000 geopackage has multiple layers; the one we want for
lithology is `geology_units25k` (others are faults/contacts/structure —
not needed for the susceptibility check).

In [ ]:
geology = gpd.read_file(str(inpath / "geopackage/TasGeology25.gpkg"), layer="geology_units25k").to_crs(TARGET_CRS)
geology_mc = gpd.clip(geology, mole_creek)

print(f"Geology polygons in Mole Creek: {len(geology_mc)}")
geology_mc.to_file(outpath / "geology_mc_GDA94.gpkg", driver="GPKG")

fig, ax = plt.subplots(figsize=(6, 6))
mole_creek.boundary.plot(ax=ax, color="black")
geology_mc.plot(ax=ax, column="description", legend=False)
plt.title("Geology units within Mole Creek (see 'description' field for lithology)")
plt.show()

## 4. DEM — merged, clipped, and used as the raster template

Mole Creek straddles the Meander Valley / Kentish municipality boundary,
so both 25m DEM tiles are merged before clipping.

**NoData handling matters here** — the source `.asc` files declare
`-9999.0` as NoData; without explicitly carrying that through `merge()`
and `mask()`, it gets treated as a real (very wrong) elevation value.

In [ ]:
dem_dir = inpath / "DEM"
candidate_tiles = ["LIST_DEM_25M_MEANDER_VALLEY", "LIST_DEM_25M_KENTISH"]
dem_files = [next((dem_dir / t).glob("*.asc")) for t in candidate_tiles]
print("DEM tiles used:", [f.name for f in dem_files])

srcs = [rasterio.open(str(f)) for f in dem_files]
dem_nodata = srcs[0].nodata
dem_mosaic, dem_transform = merge(srcs, nodata=dem_nodata)
dem_crs = srcs[0].crs
for s in srcs:
    s.close()

mosaic_profile = {
    "driver": "GTiff", "height": dem_mosaic.shape[1], "width": dem_mosaic.shape[2],
    "count": 1, "dtype": dem_mosaic.dtype, "crs": dem_crs, "transform": dem_transform,
    "nodata": dem_nodata,
}
with MemoryFile() as memfile:
    with memfile.open(**mosaic_profile) as dataset:
        dataset.write(dem_mosaic)
        dem_mc, dem_mc_transform = mask(dataset, mole_creek.geometry, crop=True, nodata=dem_nodata)

dem_mc_profile = mosaic_profile.copy()
dem_mc_profile.update(height=dem_mc.shape[1], width=dem_mc.shape[2], transform=dem_mc_transform)
with rasterio.open(outpath / "dem_mc_GDA94.tif", "w", **dem_mc_profile) as dst:
    dst.write(dem_mc)

dem_valid = np.ma.masked_equal(dem_mc, dem_nodata)
print("Mole Creek DEM clipped shape:", dem_mc.shape)
print("Elevation range (m), NoData excluded:", dem_valid.min(), "-", dem_valid.max())

plt.imshow(dem_valid[0], cmap="terrain")
plt.title("Mole Creek DEM (clipped)")
plt.colorbar(label="Elevation (m)")
plt.show()

## Stage 1 outputs

Written to `Output_data/` on the shared OneDrive:

- `karst_mc_GDA94.gpkg` — Karst Atlas polygons, clipped
- `geology_mc_GDA94.gpkg` — geology units, clipped
- `dem_mc_GDA94.tif` — merged + clipped DEM, EPSG:28355, NoData handled correctly

This DEM is the **raster template** for everything from here — any layer
that needs rasterising (e.g. the land-use pressure layer once DEA data is
in) should be resampled onto this same grid (same transform/shape/CRS).

**Next (Stage 2):** reclassify `KCATEGORY` into an ordinal karst
susceptibility score, and bring in `KEXPOSED`/`KCOVERED`/`KINTERSTR` for
the exposure-type layer.